# Jensen-Shannon divergence

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/L16B06")

## Load training and test datasets

In [ ]:
import xarray as xr

## Open a netCDF file in a xarray dataset
fname = 'data/garachico256.ens.nc'
ds    = xr.open_dataset(fname)
train = ds['tephra_col_mass']

fname = 'data/garachico2048.ens.nc'
ds    = xr.open_dataset(fname)
test  = ds['tephra_col_mass']

## Load a pre-trained VAE

In [ ]:
import torch
from modules.model import VariationalAutoencoder
from modules.dataset import MinMaxScale

## Load weight parameters and some metadata
fname = output_dir / 'model.pt'
checkpoint = torch.load(fname)

## Recreate the model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'])
model.load_state_dict(checkpoint['model_state_dict'])

## Generate a VAE ensemble

In [ ]:
## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
transform = MinMaxScale(min_value, max_value)

In [ ]:
## Generate nens new samples
def generate_samples(nens = 2048):
    z = torch.randn(nens, checkpoint['LATENT_DIM'])
    with torch.no_grad():
        new_sample = model.decode(z)
        x = transform.invert(new_sample).squeeze()
    return x.numpy()

## Compute JS divergence

In [ ]:
import numpy as np
from modules.metrics import get_js

## Data
x_train = train.values
x_test  = test.values

In [ ]:
## Training JS divergence
js_train = get_js(x_train,x_test, nbins=128, hist_range=[0,40])
js_train_max = np.percentile(js_train, 99)

In [ ]:
## VAE JS divergence
vae_sizes = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]
js_vae_list = []
for nens in vae_sizes:
    x_vae = generate_samples(nens = nens)
    js_vae = get_js(x_vae,x_test, nbins=128, hist_range=[0,40])
    js_vae_list.append(np.percentile(js_vae, 99))    

## Plot JSD

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 200

fig, ax = plt.subplots()

ax.plot([10,1E4],[js_train_max,js_train_max], 
        color     = "g", 
        linestyle = "dashdot",
        label     = 'Training dataset (256 samples)',
       )
ax.scatter(vae_sizes, js_vae_list, label = 'VAE')

ax.set(ylabel = 'Jensen–Shannon divergence (99th percentile)',
       xlabel = 'VAE-generated ensemble size',
       xscale ='log',
       ylim = [0,0.6],
       #yscale ='log',
      )

ax.annotate("256",
            xy=(256, 0.135), xycoords='data',
            xytext=(256, 0.22), textcoords='data',
            va="center", ha="center", 
            arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))

ax.legend()
ax.grid(ls='--', alpha=0.4)